# Volume 2. Sequence Memory And LRI

Question: can a short trail of assemblies be stored, inspected, and cued back into activity?

This notebook is a memory-trace inspection, not a victory lap. You will form three assemblies in order, print their overlap matrix, then enable Long-Range Inhibition (LRI) and see what recall does under one seeded parameter setting.

In [ ]:
from neural_assemblies.core.brain import Brain
from neural_assemblies.assembly_calculus import (
    chance_overlap,
    ordered_recall,
    overlap,
    sequence_memorize,
)

Set up one area and three stimuli. The values are small enough to run quickly on CPU but large enough to show nontrivial overlap behavior.

In [ ]:
N = 10_000
K = 100

brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
for i in range(3):
    brain.add_stimulus(f"s{i}", K)
brain.add_area("A", n=N, k=K, beta=0.1)

print("Chance overlap baseline:", f"{chance_overlap(K, N):.3f}")

Memorize an ordered sequence. The returned `Sequence` stores the assembly snapshot for each stimulus in the final repetition.

In [ ]:
memorized = sequence_memorize(
    brain,
    ["s0", "s1", "s2"],
    "A",
    rounds_per_step=10,
    repetitions=3,
)

print("Sequence length:", len(memorized))
print("Assembly sizes:", [len(assembly) for assembly in memorized])
print("Mean consecutive overlap:", f"{memorized.mean_consecutive_overlap():.3f}")

The overlap matrix is the notebook's little map. Diagonal values should be `1.0` because every assembly is compared with itself. Off-diagonal values show how separate the remembered items are from each other.

In [ ]:
matrix = memorized.overlap_matrix()
for row in matrix:
    print(" ".join(f"{value:.3f}" for value in row))

Enable LRI before recall. LRI suppresses recently active winners so self-projection can move away from the current assembly instead of immediately falling back into it.

In [ ]:
brain.set_lri("A", refractory_period=3, inhibition_strength=100.0)

recalled = ordered_recall(
    brain,
    "A",
    "s0",
    max_steps=10,
    known_assemblies=list(memorized),
)

print("Recalled assemblies:", len(recalled))
for i, assembly in enumerate(recalled):
    overlaps = [overlap(assembly, target) for target in memorized]
    best = max(range(len(overlaps)), key=overlaps.__getitem__)
    print(f"recall[{i}] best matches memorized[{best}] with overlap {overlaps[best]:.3f}")

Interpret the recall output cautiously. If recall stops after the cue or enters a nearby state, that is still useful information about this parameter setting. Treat stronger sequence-recall claims as research claims tied to specific experiments, not as a conclusion from this notebook alone.

Try next: change `refractory_period` or `inhibition_strength` and rerun only the recall cell. The question to ask is not "did it pass?" but "how did the trajectory change?"